In [2]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def calculate_avm_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Compute headline (MdAPE) and complementary metrics (R2, MAPE, MAE, RMSE)

    for AVM evaluation on dollar-scale targets.
    """
    abs_errors = np.abs(y_true - y_pred)
    percentage_errors = abs_errors / y_true

    mdape = float(np.median(percentage_errors) * 100)
    mape = float(np.mean(percentage_errors) * 100)
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))

    return {
        "MdAPE (%)": round(mdape, 2),
        "MAPE (%)": round(mape, 2),
        "MAE ($)": round(mae, 2),
        "RMSE ($)": round(rmse, 2),
        "R2": round(r2, 4),
    }


def train_and_evaluate_model(
    model,
    model_name: str,
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_val: pd.DataFrame,
    y_val: np.ndarray,
    X_test: pd.DataFrame,
    y_test: np.ndarray,
) -> pd.DataFrame:
    """Fits regressor on log-transformed target and evaluates on raw dollar scale

    across Train, Validation, and Test chronological splits.
    """
    # 1. Fit strictly on log-transformed training target to stabilize target variance
    # across multi-scale price points ($150K to $5M+)
    y_train_log = np.log1p(y_train)
    model.fit(X_train, y_train_log)

    # 2. Predict and invert back to raw dollar scale via expm1
    y_train_pred = np.expm1(model.predict(X_train))
    y_val_pred = np.expm1(model.predict(X_val))
    y_test_pred = np.expm1(model.predict(X_test))

    # 3. Calculate headline (MdAPE) and complementary metrics across all 3 splits
    metrics_train = calculate_avm_metrics(y_train, y_train_pred)
    metrics_val = calculate_avm_metrics(y_val, y_val_pred)
    metrics_test = calculate_avm_metrics(y_test, y_test_pred)

    df = pd.DataFrame(
        [metrics_train, metrics_val, metrics_test],
        index=["Train", "Validation", "Test"],
    )
    df.insert(0, "Model", model_name)
    return df


def run_week5_model_comparison(data_dir: str | Path, output_dir: str | Path) -> pd.DataFrame:
    """Trains Baseline Linear Regression, Decision Tree, and Random Forest models,

    compares multi-split performance, and exports model artifacts.
    """
    data_dir = Path(data_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Load processed features and target arrays (chronologically partitioned)
    X_train = pd.read_parquet(data_dir / "X_train_processed.parquet")
    X_val = pd.read_parquet(data_dir / "X_val_processed.parquet")
    X_test = pd.read_parquet(data_dir / "X_test_processed.parquet")

    y_train = pd.read_parquet(data_dir / "y_train.parquet")["ClosePrice"].to_numpy()
    y_val = pd.read_parquet(data_dir / "y_val.parquet")["ClosePrice"].to_numpy()
    y_test = pd.read_parquet(data_dir / "y_test.parquet")["ClosePrice"].to_numpy()

    # 2. Instantiate model candidates with explicit parameter hyperparameter choices
    models = {
        "Linear_Regression_Baseline": LinearRegression(),
        
        "Decision_Tree_Unconstrained": DecisionTreeRegressor(
            # random_state=42: Fixes the pseudo-random seed during greedy split selection
            # (when features have identical impurity gains), ensuring bit-for-bit reproducible trees.
            random_state=42
        ),
        
        "Decision_Tree_Regularized": DecisionTreeRegressor(
            # max_depth=12: Caps tree depth to 12 levels. Without a depth limit, decision trees grown
            # on housing data expand until leaves are pure, effectively memorizing individual high-end
            # sales or spatial noise. Depth 12 allows full interaction hierarchy (e.g., ZIP -> LivingArea -> Age)
            # while truncating deep overfitted branches.
            max_depth=12,
            
            # min_samples_leaf=20: Requires at least 20 historical comparable transactions in every leaf node.
            # In real estate AVMs, this prevents the tree from isolating extreme price outliers (e.g., luxury
            # custom builds or unverified non-arm's-length sales) into single-observation leaves, forcing
            # leaf valuations to be derived from a statistically meaningful local comp pool.
            min_samples_leaf=20,
            
            # random_state=42: Enforces deterministic tree growth across identical data snapshots.
            random_state=42,
        ),
        
        "Random_Forest_Default": RandomForestRegressor(
            # n_estimators=100: Builds 100 decorrelated decision trees. Averaging across 100 bootstrapped
            # samples substantially suppresses ensemble prediction variance without introducing extra bias.
            n_estimators=100,
            
            # max_depth=20: Allows deeper individual tree capacity (20 levels) than a single decision tree
            # because bootstrap aggregation (bagging) and random feature sub-selection (mtry) inherently
            # protect the ensemble against overfitting while capturing complex spatial/property interactions.
            max_depth=20,
            
            # min_samples_leaf=5: Sets a moderate leaf smoothing threshold. Smaller than a single tree (5 vs 20)
            # because the ensemble averages out leaf variance across trees, allowing fine-grained sensitivity
            # to micro-neighborhood pricing boundaries.
            min_samples_leaf=5,
            
            # n_jobs=-1: Parallelizes tree construction across all CPU cores for scalable fitting on large MLS sets.
            n_jobs=-1,
            
            # random_state=42: Pins pseudo-random seed governing row bootstrap sampling and random feature
            # subspace selection at each node split, ensuring strict pipeline reproducibility.
            random_state=42,
        ),
    }

    results_list = []
    for model_name, model in models.items():
        res = train_and_evaluate_model(
            model, model_name, X_train, y_train, X_val, y_val, X_test, y_test
        )
        results_list.append(res)

    # 3. Combine into master multi-level comparison DataFrame
    comparison_df = pd.concat(results_list)

    # Extract test-set summary table for executive review
    test_summary = comparison_df.xs("Test", level=0 if isinstance(comparison_df.index, pd.MultiIndex) else None) if False else comparison_df[comparison_df.index == "Test"]

    # 4. Save comprehensive multi-split results to JSON and test summary to CSV
    summary_path = output_dir / "04_model_comparison_summary.csv"
    test_summary.to_csv(summary_path, index=False)

    with open(output_dir / "04_model_comparison_full.json", "w") as f:
        json.dump(comparison_df.reset_index().to_dict(orient="records"), f, indent=4)

    print("=== Multi-Split Model Performance (Train vs. Validation vs. Test) ===")
    print(comparison_df.to_string())

    return comparison_df


if __name__ == "__main__":
    DATA_DIR = Path("model_artifacts")
    OUTPUT_DIR = Path("models/week5_comparison")
    run_week5_model_comparison(DATA_DIR, OUTPUT_DIR)

=== Multi-Split Model Performance (Train vs. Validation vs. Test) ===
                                  Model  MdAPE (%)  MAPE (%)    MAE ($)    RMSE ($)      R2
Train        Linear_Regression_Baseline      11.94     16.31  212569.38   630273.28  0.5243
Validation   Linear_Regression_Baseline      12.23     16.50  227634.47  1002464.16 -0.1681
Test         Linear_Regression_Baseline      12.16     16.31  223498.65   517147.61  0.7071
Train       Decision_Tree_Unconstrained       0.00      0.00      31.92     1356.05  1.0000
Validation  Decision_Tree_Unconstrained      12.86     18.70  248269.14   475464.15  0.7372
Test        Decision_Tree_Unconstrained      13.24     18.76  252317.98   476257.13  0.7516
Train         Decision_Tree_Regularized       9.89     13.61  170808.52   320254.68  0.8772
Validation    Decision_Tree_Regularized      11.06     15.08  201503.30   385148.37  0.8276
Test          Decision_Tree_Regularized      11.30     15.38  210512.63   399813.94  0.8249
Train     

Linear Regression - Basline.
Ordinary Least Squares (OLS) minimizes mean squared error, making it extremely sensitive to unhandled luxury price outliers or spatial volatility in the validation split.

Decision Tree Unconstrained- Worse than baseline. 
Weakness: Severe Overfitting: The unconstrained tree splits down to pure leaf nodes containing single training transactions.

Decision Tree Regularized- Better than baseline. 
Weakness: A single decision tree creates rigid axis-aligned step functions across continuous spatial features (latitude/longitude), leading to sharp price cliffs between adjacent neighborhoods.   

Random Forest - Better than baseline.
Weakness: Residual overfitting remains, with a Train MdAPE of 6.07% versus a Test MdAPE of 9.48% (a 3.41 percentage point gap). Default tree depths allow individual trees in the forest to overfit local noise